# 2. Feature Engineering for Grafana Logs

This notebook prepares features from Grafana logs for clustering analysis.

## Objectives
1. Extract text features from logs
2. Create semantic representations
3. Engineer temporal features
4. Build composite feature vectors
5. Prepare data for embedding models

In [ ]:
import json
import pandas as pd
import numpy as np
import re
from datetime import datetime
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

## 1. Load Data

In [ ]:
def load_grafana_logs(file_path):
    logs = []
    with open(file_path, 'r') as f:
        for line in f:
            try:
                logs.append(json.loads(line.strip()))
            except json.JSONDecodeError:
                continue
    return logs

log_file = '../output/grafana/logs_2024-01-01.jsonl'
print(f"Loading logs from {log_file}...")
logs = load_grafana_logs(log_file)
print(f"✅ Loaded {len(logs):,} log entries")

## 2. Text Feature Extraction

In [ ]:
def extract_text_features(log):
    """
    Extract text-based features from a log entry.
    """
    # Combine relevant text fields
    text_parts = []
    
    # Dashboard and panel info
    dashboard = log.get('dashboard', '')
    panel_title = log.get('panel', {}).get('title', '')
    
    # Query information
    query_expr = log.get('target', {}).get('expr', '')
    legend_format = log.get('target', {}).get('legendFormat', '')
    
    # Service tags
    service = log.get('tags', {}).get('service', '')
    
    # Combine into meaningful text
    text_parts = [
        f"dashboard: {dashboard}",
        f"panel: {panel_title}",
        f"service: {service}",
        f"query: {query_expr}",
        f"legend: {legend_format}"
    ]
    
    combined_text = " | ".join(text_parts)
    
    return {
        'combined_text': combined_text,
        'dashboard': dashboard,
        'panel_title': panel_title,
        'service': service,
        'query_expr': query_expr,
        'legend_format': legend_format
    }

# Extract text features for all logs
print("Extracting text features...")
text_features = [extract_text_features(log) for log in logs]
print(f"✅ Extracted text features for {len(text_features):,} logs")

# Show example
print(f"\nExample combined text:")
print(text_features[0]['combined_text'][:200] + "...")

## 3. Semantic Placeholder Replacement

Replace specific values with placeholders for better generalization.

In [ ]:
def apply_semantic_placeholders(text):
    """
    Replace specific values with semantic placeholders.
    """
    # Replace numbers
    text = re.sub(r'\b\d+\.\d+\b', '<NUMBER>', text)
    text = re.sub(r'\b\d+\b', '<NUMBER>', text)
    
    # Replace pod names (service-name-###-xxxxx)
    text = re.sub(r'\b([a-z]+-[a-z]+)-\d+-[a-z0-9]+\b', r'\1-<POD_ID>', text)
    
    # Replace timestamps
    text = re.sub(r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}', '<TIMESTAMP>', text)
    
    # Replace UUIDs
    text = re.sub(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', '<UUID>', text)
    
    # Replace IP addresses
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP_ADDRESS>', text)
    
    return text

# Apply placeholders
print("Applying semantic placeholders...")
for feat in text_features:
    feat['normalized_text'] = apply_semantic_placeholders(feat['combined_text'])

print("✅ Applied semantic placeholders")
print(f"\nExample normalized text:")
print(text_features[0]['normalized_text'][:200] + "...")

## 4. Temporal Features

In [ ]:
def extract_temporal_features(log):
    """
    Extract temporal features from log entry.
    """
    datapoints = log.get('datapoints', [[None, None]])
    if datapoints and datapoints[0][1]:
        timestamp_ms = datapoints[0][1]
        dt = datetime.fromtimestamp(timestamp_ms / 1000.0)
        
        return {
            'timestamp': dt,
            'hour': dt.hour,
            'minute': dt.minute,
            'day_of_week': dt.weekday(),
            'is_business_hours': 9 <= dt.hour <= 17,
            'is_weekend': dt.weekday() >= 5,
            'time_of_day': 'night' if dt.hour < 6 else 'morning' if dt.hour < 12 else 'afternoon' if dt.hour < 18 else 'evening'
        }
    return {}

# Extract temporal features
print("Extracting temporal features...")
temporal_features = [extract_temporal_features(log) for log in logs]
print(f"✅ Extracted temporal features for {len(temporal_features):,} logs")

## 5. Metric Value Features

In [ ]:
def extract_metric_features(log):
    """
    Extract metric value features.
    """
    datapoints = log.get('datapoints', [[None, None]])
    value = datapoints[0][0] if datapoints else None
    
    if value is not None:
        return {
            'value': value,
            'value_log': np.log1p(abs(value)) if value > 0 else 0,
            'value_category': 'low' if value < 33 else 'medium' if value < 66 else 'high'
        }
    return {'value': None, 'value_log': None, 'value_category': 'unknown'}

# Extract metric features
print("Extracting metric features...")
metric_features = [extract_metric_features(log) for log in logs]
print(f"✅ Extracted metric features for {len(metric_features):,} logs")

## 6. Categorical Features

In [ ]:
def extract_categorical_features(log):
    """
    Extract categorical features.
    """
    return {
        'panel_type': log.get('panel', {}).get('type', 'unknown'),
        'datasource': log.get('panel', {}).get('datasource', 'unknown'),
        'viz_type': log.get('meta', {}).get('preferredVisualisationType', 'unknown'),
        'environment': log.get('tags', {}).get('environment', 'unknown'),
        'cluster': log.get('tags', {}).get('cluster', 'unknown')
    }

# Extract categorical features
print("Extracting categorical features...")
categorical_features = [extract_categorical_features(log) for log in logs]
print(f"✅ Extracted categorical features for {len(categorical_features):,} logs")

## 7. Combine All Features

In [ ]:
# Combine all features into a single DataFrame
print("Combining all features...")

combined_features = []
for i in range(len(logs)):
    feature_dict = {
        **text_features[i],
        **temporal_features[i],
        **metric_features[i],
        **categorical_features[i]
    }
    combined_features.append(feature_dict)

df = pd.DataFrame(combined_features)
print(f"\n✅ Created feature DataFrame with shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 8. Create Log Templates

Group similar logs by their normalized text patterns.

In [ ]:
# Create templates from normalized text
template_counts = df['normalized_text'].value_counts()

print(f"\n📊 Template Statistics:")
print(f"Total unique templates: {len(template_counts):,}")
print(f"Most common template appears {template_counts.values[0]:,} times")
print(f"\nTop 10 Templates:")
for i, (template, count) in enumerate(template_counts.head(10).items(), 1):
    print(f"{i:2d}. [{count:5,}x] {template[:100]}...")

## 9. Feature Statistics

In [ ]:
print("\n" + "="*80)
print("FEATURE STATISTICS")
print("="*80)

print(f"\n📊 Categorical Features:")
print(f"  - Unique services: {df['service'].nunique()}")
print(f"  - Unique dashboards: {df['dashboard'].nunique()}")
print(f"  - Unique panel titles: {df['panel_title'].nunique()}")
print(f"  - Unique panel types: {df['panel_type'].nunique()}")
print(f"  - Unique environments: {df['environment'].nunique()}")

print(f"\n⏱️  Temporal Features:")
if 'hour' in df.columns:
    print(f"  - Hour range: {df['hour'].min()} - {df['hour'].max()}")
    print(f"  - Business hours logs: {df['is_business_hours'].sum():,} ({df['is_business_hours'].sum()/len(df)*100:.1f}%)")
    print(f"  - Weekend logs: {df['is_weekend'].sum():,} ({df['is_weekend'].sum()/len(df)*100:.1f}%)")

print(f"\n📈 Metric Features:")
if 'value' in df.columns:
    print(f"  - Value range: [{df['value'].min():.2f}, {df['value'].max():.2f}]")
    print(f"  - Mean value: {df['value'].mean():.2f}")
    print(f"  - Std value: {df['value'].std():.2f}")

print("\n" + "="*80)

## 10. Prepare for Clustering

Create the final feature set for clustering.

In [ ]:
# Create a clean dataset for clustering
clustering_data = df[[
    'normalized_text',
    'dashboard',
    'panel_title',
    'service',
    'panel_type',
    'value',
    'value_category'
]].copy()

# Add row index
clustering_data['log_id'] = range(len(clustering_data))

print(f"\n✅ Prepared clustering data: {clustering_data.shape}")
clustering_data.head()

## 11. Save Engineered Features

In [ ]:
# Save full feature set
output_file = '../output/grafana/engineered_features.parquet'
df.to_parquet(output_file, index=False)
print(f"✅ Saved full features to: {output_file}")

# Save clustering-ready data
clustering_file = '../output/grafana/clustering_features.parquet'
clustering_data.to_parquet(clustering_file, index=False)
print(f"✅ Saved clustering features to: {clustering_file}")

# Save normalized texts separately (for embedding)
text_file = '../output/grafana/normalized_texts.txt'
with open(text_file, 'w') as f:
    for text in df['normalized_text']:
        f.write(text + '\n')
print(f"✅ Saved normalized texts to: {text_file}")

print(f"\n" + "="*80)
print("✅ Feature Engineering Complete")
print("="*80)
print(f"\nNext step: Run notebook 3 for embedding generation")